In [ ]:
# qsvm_model.py

import numpy as onp  # normal numpy

import pennylane as qml
from pennylane import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score


# -----------------
# 1. Load processed data
# -----------------
def load_breast_cancer_processed():
    train = onp.load("data/processed/bc_train.npz")
    val   = onp.load("data/processed/bc_val.npz")
    test  = onp.load("data/processed/bc_test.npz")

    X_train, y_train = train["x"], train["y"]
    X_val,   y_val   = val["x"],   val["y"]
    X_test,  y_test  = test["x"],  test["y"]

    return X_train, y_train, X_val, y_val, X_test, y_test


# -----------------
# 2. Device and feature map (same as VQC encoding)
# -----------------
n_qubits = 2
n_features = 4

dev = qml.device("default.qubit", wires=n_qubits, shots=None)

def angle_encode_2q_4f(x):
    x = np.array(x).ravel()
    assert len(x) == 4, f"Expected 4 features, got {len(x)}"
    angles = np.pi * x

    # qubit 0
    qml.RY(angles[0], wires=0)
    qml.RZ(angles[1], wires=0)

    # qubit 1
    qml.RY(angles[2], wires=1)
    qml.RZ(angles[3], wires=1)


@qml.qnode(dev)
def feature_state(x):
    """
    Prepare the quantum state |psi(x)> using only the feature map.
    Return the full statevector.
    """
    angle_encode_2q_4f(x)
    return qml.state()


# -----------------
# 3. Build kernel matrix
# -----------------
def compute_feature_states(X):
    """
    X: shape (N, 4)
    Returns: states, shape (N, 2**n_qubits)
    """
    states = [feature_state(x) for x in X]
    states = np.stack(states)
    return states


def kernel_matrix_from_states(states_A, states_B):
    """
    states_A: shape (NA, D)
    states_B: shape (NB, D)
    K_ij = | <psi_A_i | psi_B_j> |^2
    """
    # convert to normal numpy for matrix multiply
    A = onp.array(states_A)
    B = onp.array(states_B)

    # inner products (NA x NB)
    inner = A @ B.conj().T   # complex matrix
    K = onp.abs(inner) ** 2  # elementwise square modulus
    return K


# -----------------
# 4. Train QSVM
# -----------------
def train_qsvm(C=1.0):
    X_train, y_train, X_val, y_val, X_test, y_test = load_breast_cancer_processed()

    print("Computing quantum feature states...")
    states_train = compute_feature_states(X_train)
    states_val   = compute_feature_states(X_val)
    states_test  = compute_feature_states(X_test)

    print("Building kernel matrices...")
    K_train = kernel_matrix_from_states(states_train, states_train)  # (n_train, n_train)
    K_val   = kernel_matrix_from_states(states_val,   states_train)  # (n_val, n_train)
    K_test  = kernel_matrix_from_states(states_test,  states_train)  # (n_test, n_train)

    # SVM with precomputed kernel
    clf = SVC(kernel="precomputed", C=C)
    clf.fit(K_train, y_train)

    # predictions
    y_val_pred  = clf.predict(K_val)
    y_test_pred = clf.predict(K_test)

    val_acc  = accuracy_score(y_val,  y_val_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    test_f1  = f1_score(y_test, y_test_pred)

    print("QSVM (quantum kernel SVM):")
    print("  Val accuracy:", val_acc)
    print("  Test accuracy:", test_acc)
    print("  Test F1:", test_f1)

    return clf


if __name__ == "__main__":
    train_qsvm()


Computing quantum feature states...
Building kernel matrices...
QSVM (quantum kernel SVM):
  Val accuracy: 0.631578947368421
  Test accuracy: 0.6228070175438597
  Test F1: 0.7675675675675676
